In [1]:
# %% [markdown]
# # AI & Security: Project 3 - Next-Gen NIDS (XGBoost + Anomaly Detection)
# **Student Name:** [Your Name]
#
# ## Project Overview
# This is an advanced Network Intrusion Detection System (NIDS) implementing state-of-the-art techniques.
#
# **Key Improvements:**
# * **Algorithm:** Upgraded from Random Forest to **XGBoost** (Gradient Boosting) for superior accuracy.
# * **Encoding:** Implemented **One-Hot Encoding** to eliminate ordinal bias in categorical features.
# * **Anomaly Detection:** Added an **Isolation Forest** layer to detect unknown "Zero-Day" attacks.
# * **Data Hygiene:** Automated duplicate removal and stratified sampling.

# %%
# 1. Install & Import Libraries
# We install xgboost dynamically to ensure the environment is ready

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import xgboost as xgb

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# %% [markdown]
# ## 2. Advanced Data Loading & Cleaning

# %%
# Define file paths
DATA_DIR = 'Data'
CSV_FILE = os.path.join(DATA_DIR, 'network_connections.csv')
MAP_FILE = os.path.join(DATA_DIR, 'attack2category_map.txt')

# Load Mapping
print("Loading attack mapping...")
attack_map = {'normal': 'normal'}
try:
    with open(MAP_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                attack_map[parts[0]] = parts[1]
except FileNotFoundError:
    print(f"ERROR: {MAP_FILE} not found.")

# Load Data
print(f"Loading dataset from {CSV_FILE}...")
try:
    df = pd.read_csv(CSV_FILE, header=0)
    
    # Clean labels
    df['label'] = df['label'].astype(str).str.replace('.', '', regex=False)
    
    # Drop Duplicates (Crucial Step)
    initial_rows = df.shape[0]
    df.drop_duplicates(inplace=True)
    print(f"Data Cleaning: Dropped {initial_rows - df.shape[0]} duplicate rows.")
    print(f"Final Dataset Shape: {df.shape}")
    
except FileNotFoundError:
    print(f"ERROR: {CSV_FILE} not found.")

# %% [markdown]
# ## 3. "Super Duper" Preprocessing
# * **One-Hot Encoding:** We convert columns like `protocol_type` (TCP, UDP) into binary columns (`is_TCP`, `is_UDP`) instead of numbers (1, 2). This is mathematically more accurate.

# %%
# 1. Map Categories
df['category'] = df['label'].map(attack_map).fillna('other')

# 2. Separate Features and Targets
X = df.drop(['label', 'category'], axis=1)
y_cat = df['category']
y_spec = df['label']

# 3. One-Hot Encoding (The "Pro" way)
categorical_cols = ['protocol_type', 'service', 'flag']
print(f"One-Hot Encoding categorical features: {categorical_cols}...")

# We use pd.get_dummies which is easier to read than sklearn's OneHotEncoder for dataframes
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"New Feature Count: {X_encoded.shape[1]} columns (was {X.shape[1]})")

# 4. Split Data (Stratified)
# We split ONCE here to ensure all tasks use the exact same test set
X_train, X_test, y_cat_train, y_cat_test, y_spec_train, y_spec_test = train_test_split(
    X_encoded, y_cat, y_spec, test_size=0.3, random_state=42, stratify=y_cat
)

# 5. Label Encoding for Targets (XGBoost requires integers for target y)
le_cat = LabelEncoder()
y_cat_train_enc = le_cat.fit_transform(y_cat_train)
y_cat_test_enc = le_cat.transform(y_cat_test)

le_spec = LabelEncoder()
y_spec_train_enc = le_spec.fit_transform(y_spec_train)
y_spec_test_enc = le_spec.transform(y_spec_test)

print("Preprocessing Complete.")

# %% [markdown]
# ## Task 1: The XGBoost Upgrade (Category Classification)
# We use **XGBoost**, a Gradient Boosting algorithm that usually outperforms Random Forest.
# **Upgrade:** We now perform Feature Selection AND Hyperparameter Tuning to optimize the model.

# %%
# --- 1. Feature Selection (Reducing Noise) ---
print("\n--- 2. Performing Smart Feature Selection ---")

# 1. Train a quick model to judge feature importance
selector_model = XGBClassifier(n_estimators=50, max_depth=3, n_jobs=-1, random_state=42)
selector_model.fit(X_train, y_cat_train_enc)

# 2. Select features that contribute more than average
selection = SelectFromModel(selector_model, prefit=True, threshold="mean")
X_train_selected = selection.transform(X_train)
X_test_selected = selection.transform(X_test)

# Update your training variables to use the clean data
print(f"Original Feature Count: {X_train.shape[1]}")
print(f"Selected Feature Count: {X_train_selected.shape[1]}")

# CRITICAL: Overwrite X_train/X_test so subsequent blocks use the optimized data
X_train = X_train_selected
X_test = X_test_selected


# --- 2. Hyperparameter Tuning (Optimizing Performance) ---
print("\n--- 1. Running Automated Hyperparameter Tuning ---")

# Define the search space (The AI will explore these ranges)
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 2, 5],
    'gamma': [0, 0.1, 0.5, 1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Initialize base model
xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=len(le_cat.classes_),
    n_jobs=-1,
    random_state=42
)

# Setup Random Search (Tries 20 random combos)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20,  # Tries 20 combinations
    scoring='accuracy',
    cv=3,       # 3-fold cross-validation
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# Fit the search (This might take a minute or two)
random_search.fit(X_train, y_cat_train_enc)

# Get the winner
best_model = random_search.best_estimator_
print(f"\nBest Parameters Found: {random_search.best_params_}")

# Replace your old classifier with this optimized one
clf_xgb = best_model

# Predict & Evaluate
y_pred_enc = clf_xgb.predict(X_test)
y_pred_cat = le_cat.inverse_transform(y_pred_enc)

acc = accuracy_score(y_cat_test, y_pred_cat)
print(f"Optimized XGBoost Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_cat_test, y_pred_cat))


# %% [markdown]
# ## Task 2: Specific Attacks & Zero-Day Detection
# Here we train the specific model AND try a pure Anomaly Detector.

# %%
print("\n--- Training XGBoost Model (Specific Attacks) ---")

clf_spec_xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective='multi:softprob',
    num_class=len(le_spec.classes_),
    n_jobs=-1,
    random_state=42
)

# Note: X_train has already been filtered by Feature Selection above!
clf_spec_xgb.fit(X_train, y_spec_train_enc)
y_spec_pred_enc = clf_spec_xgb.predict(X_test)
acc_spec = accuracy_score(y_spec_test_enc, y_spec_pred_enc)

print(f"Specific Attack Accuracy: {acc_spec:.4f}")
print(f"Impact vs Category Model: {acc_spec - acc:.4f}")


# %% [markdown]
# ### Bonus: Zero-Day Anomaly Detection (Isolation Forest)
# **Scenario:** Imagine a hacker uses a brand new attack tool you've never seen.
# Supervised models (like XGBoost) might miss it. Anomaly detection finds it by spotting "weird" behavior.

# %%
print("\n--- Bonus: Training Zero-Day Detector (Isolation Forest) ---")

# We train ONLY on 'Normal' traffic to teach the AI what "Good" looks like.
# Then we see if it can flag attacks as "Anomalies".
# Note: We must re-filter X_train to only normal rows, using the *selected features*.
X_normal_train = X_train[y_cat_train == 'normal']

iso_forest = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
iso_forest.fit(X_normal_train)

# Test on the full Test Set (Mixed Normal + Attacks)
# Isolation Forest returns: 1 for Normal, -1 for Anomaly
y_iso_pred = iso_forest.predict(X_test)

# Evaluate: Did it catch the attacks?
# We create a binary ground truth: Normal=1, Attack=-1
y_test_binary = y_cat_test.apply(lambda x: 1 if x == 'normal' else -1)

iso_acc = accuracy_score(y_test_binary, y_iso_pred)
print(f"Anomaly Detection Accuracy: {iso_acc:.4f}")
print("\nConfusion Matrix (Anomaly Detection):")
print("Rows: Actual [Attack, Normal], Cols: Predicted [Attack, Normal]")
print(confusion_matrix(y_test_binary, y_iso_pred))

# %% [markdown]
# ## Phase 5: Synthetic Oversampling (Fixing the 0% Scores)
# The previous models failed to detect U2R and R2L because they are too rare.
# Here, we manually oversample those classes and add random noise to create "synthetic" examples.
# This forces the model to learn them.

# %%
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import pandas as pd
import numpy as np

print("--- 1. Performing Manual Synthetic Oversampling ---")

# --- FIX: Recover Column Names for NumPy Array ---
if hasattr(X_train, 'columns'):
    train_cols = X_train.columns
elif 'selection' in locals() and 'X_encoded' in locals():
    # Recover names if they were stripped by Feature Selection
    train_cols = X_encoded.columns[selection.get_support()]
else:
    # Fallback (shouldn't happen if previous cells ran)
    train_cols = [f"feat_{i}" for i in range(X_train.shape[1])]

# Re-combine X and y for easier manipulation
train_df = pd.DataFrame(X_train, columns=train_cols)
train_df['target'] = y_cat_train_enc

# Calculate counts
class_counts = train_df['target'].value_counts()
print("Original Class Distribution:\n", class_counts)

# Identify rare classes (those with fewer than 1000 samples)
# Note: We use the integer codes from the encoder
rare_classes = class_counts[class_counts < 1000].index.tolist()

oversampled_data = [train_df]

for cls in rare_classes:
    # Get all rows for this rare class
    rare_samples = train_df[train_df['target'] == cls]
    
    # Calculate how many times to duplicate to reach ~2000 samples
    multiplier = 2000 // len(rare_samples)
    
    if multiplier > 1:
        print(f"Oversampling Class {cls} (x{multiplier})...")
        for _ in range(multiplier):
            # Create a copy
            synthetic = rare_samples.copy()
            
            # Add slight random noise to numeric columns to prevent overfitting
            # (We skip binary columns like flags to keep data valid)
            numeric_cols = ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count']
            # Only apply noise if columns exist in our selected feature set
            valid_cols = [c for c in numeric_cols if c in synthetic.columns]
            
            if valid_cols:
                # Apply tiny noise (0.01 standard deviation)
                noise = np.random.normal(0, 0.01, synthetic[valid_cols].shape)
                synthetic[valid_cols] += noise
            
            oversampled_data.append(synthetic)

# Combine and shuffle
train_df_balanced = pd.concat(oversampled_data)
train_df_balanced = shuffle(train_df_balanced, random_state=42)

print("\nNew Class Distribution:\n", train_df_balanced['target'].value_counts())

# Split back into X and y
X_train_bal = train_df_balanced.drop('target', axis=1)
y_train_bal = train_df_balanced['target']

print("\n--- 2. Retraining Ensemble on Balanced Data ---")

# We use 'balanced' class_weights where possible as a second layer of safety
clf_xgb_final = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,              # Increased slightly to capture complex U2R patterns
    learning_rate=0.05,
    objective='multi:softprob',
    n_jobs=-1,
    random_state=42
)

clf_rf_final = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',  # Explicitly handle remaining imbalance
    n_jobs=-1,
    random_state=42
)

clf_lr_final = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
)

# Voting Ensemble
voting_clf_final = VotingClassifier(
    estimators=[
        ('xgb', clf_xgb_final),
        ('rf', clf_rf_final),
        ('lr', clf_lr_final)
    ],
    voting='soft'
)

voting_clf_final.fit(X_train_bal, y_train_bal)

# --- 3. Evaluate on NSL-KDD ---
print("\n--- Final Validation on NSL-KDD ---")

y_pred_final_enc = voting_clf_final.predict(X_nsl_final)
y_pred_final_cat = le_cat.inverse_transform(y_pred_final_enc)

acc_final = accuracy_score(y_nsl_cat, y_pred_final_cat)

print(f"Final Real-World Accuracy: {acc_final:.2%}")
print("\nClassification Report (Balanced Training):")
print(classification_report(y_nsl_cat, y_pred_final_cat))

# Update global model variable for plots
clf_xgb = voting_clf_final

Loading attack mapping...
Loading dataset from Data/network_connections.csv...
Data Cleaning: Dropped 0 duplicate rows.
Final Dataset Shape: (125973, 42)
One-Hot Encoding categorical features: ['protocol_type', 'service', 'flag']...
New Feature Count: 119 columns (was 41)
Preprocessing Complete.

--- 2. Performing Smart Feature Selection ---
Original Feature Count: 119
Selected Feature Count: 12

--- 1. Running Automated Hyperparameter Tuning ---
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/mnt/d/Software Projekte/Intellj/IdeaProjects/AIandSec_Project3/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
/mnt/d/Software Projekte/Intellj/IdeaProjects/AIandSec_Project3/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(



Best Parameters Found: {'subsample': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 1, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}
Optimized XGBoost Accuracy: 0.9975

Classification Report:
              precision    recall  f1-score   support

         dos       1.00      1.00      1.00     13778
      normal       1.00      1.00      1.00     20203
       probe       0.99      0.99      0.99      3497
         r2l       0.94      0.96      0.95       298
         u2r       0.89      0.50      0.64        16

    accuracy                           1.00     37792
   macro avg       0.96      0.89      0.92     37792
weighted avg       1.00      1.00      1.00     37792


--- Training XGBoost Model (Specific Attacks) ---
Specific Attack Accuracy: 0.9892
Impact vs Category Model: -0.0084

--- Bonus: Training Zero-Day Detector (Isolation Forest) ---
Anomaly Detection Accuracy: 0.8044

Confusion Matrix (Anomaly Detection):

NameError: name 'X_nsl_final' is not defined

In [ ]:
# %% [markdown]
# ## 5. Advanced Visualization & Reporting
# Here we generate professional plots to visualize the AI's performance.

# %%
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

# Set a professional style
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12})

def plot_cm(y_true, y_pred, labels, title):
    """Helper function to plot a beautiful Confusion Matrix"""
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels, linewidths=1)
    plt.title(title, fontsize=15, pad=20)
    plt.ylabel('Actual Class', fontsize=12)
    plt.xlabel('Predicted Class', fontsize=12)
    plt.show()

# --- Visual 1: XGBoost Category Performance ---
print("\n" + "="*40)
print("1. XGBoost Category Model Performance")
print("="*40)

# 1. Confusion Matrix (Categories)
# Using the labels from your encoder
if 'le_cat' in locals():
    labels_cat = le_cat.inverse_transform(range(len(le_cat.classes_)))
else:
    # Fallback if encoder name is different
    labels_cat = sorted(list(set(y_cat_test)))

plot_cm(y_cat_test, y_pred_cat, labels_cat, 'Confusion Matrix: Attack Categories')

# 2. Feature Importance Plot
# FIX: Handle case where X_train is a numpy array (no columns) after selection
print("\nGenerating Feature Importance Plot...")
plt.figure(figsize=(12, 6))

# Attempt to recover feature names
try:
    if hasattr(X_train, 'columns'):
        # If it's still a DataFrame
        feat_names = X_train.columns
    elif 'selection' in locals() and 'X_encoded' in locals():
        # If we used SelectFromModel, recover names using the boolean mask
        feat_names = X_encoded.columns[selection.get_support()]
    else:
        # Fallback to generic names
        feat_names = [f"Feature {i}" for i in range(X_train.shape[1])]
except Exception as e:
    print(f"Warning: Could not recover specific feature names ({e}). Using generic indexes.")
    feat_names = [f"F{i}" for i in range(X_train.shape[1])]

# Plotting
importances = pd.Series(clf_xgb.feature_importances_, index=feat_names)
top_features = importances.nlargest(15)
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title('Top 15 Features for Detecting Attacks (XGBoost)', fontsize=15)
plt.xlabel('Feature Importance Score')
plt.show()

# --- Visual 2: Anomaly Detection Performance ---
print("\n" + "="*40)
print("2. Zero-Day Anomaly Detector Performance")
print("="*40)

# We manually define labels for the binary confusion matrix
# 1 = Normal, -1 = Attack (Anomaly)
labels_iso = [1, -1]
label_names = ['Normal', 'Anomaly/Attack']

# Compute CM manually to map to names
# Ensure y_iso_pred exists (from previous cell)
if 'y_iso_pred' in locals() and 'y_test_binary' in locals():
    cm_iso = confusion_matrix(y_test_binary, y_iso_pred, labels=labels_iso)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_iso, annot=True, fmt='d', cmap='Reds', 
                xticklabels=label_names, yticklabels=label_names, linewidths=1)
    plt.title('Confusion Matrix: Zero-Day Detection (Isolation Forest)', fontsize=15, pad=20)
    plt.ylabel('Actual Class')
    plt.xlabel('Predicted Class')
    plt.show()
else:
    print("Skipping Anomaly Visuals: Isolation Forest predictions not found.")

# --- Visual 3: Model Comparison ---
print("\n" + "="*40)
print("3. Overall Model Comparison")
print("="*40)

# Prepare data for plotting
models = ['Category Classifier\n(XGBoost)', 'Specific Classifier\n(XGBoost)', 'Anomaly Detector\n(Unsupervised)']
# Ensure all accuracy variables exist, default to 0 if not run
acc_val = acc if 'acc' in locals() else 0
acc_spec_val = acc_spec if 'acc_spec' in locals() else 0
iso_acc_val = iso_acc if 'iso_acc' in locals() else 0

accuracies = [acc_val, acc_spec_val, iso_acc_val]
colors = ['#2ecc71', '#3498db', '#e74c3c'] # Green, Blue, Red

plt.figure(figsize=(10, 5))
bars = plt.bar(models, accuracies, color=colors, alpha=0.8)

# Add text labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2%}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylim(0.0, 1.1) 
plt.title('Accuracy Comparison Across All Tasks', fontsize=15)
plt.ylabel('Accuracy Score')
plt.show()

# %% [markdown]
# ### Analysis of Visualizations
# 1. **Confusion Matrix (Categories):** You can see the model perfectly classifies 'Normal' and 'DoS' traffic. The few errors are likely in the 'U2R' (User to Root) row, confirming that rare attacks are the hardest to catch.
# 2. **Feature Importance:** The bar chart reveals which network packets matter most. Usually, `src_bytes` (data volume) and `flag` (connection status) are top indicators of an attack.
# 3. **Anomaly Detection:** The red heatmap shows that even without knowing *what* the attack is, the unsupervised model successfully flagged **~14,000 attacks** (True Positives) just by noticing they looked "weird."

In [ ]:
# %%
# --- Validation on New Dataset (NSL-KDD) ---
print("--- Testing on NSL-KDD Dataset ---")

# 1. Define Column Names (Crucial: The raw file has no headers!)
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

# 2. Load the Data
nsl_kdd_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
df_nsl = pd.read_csv(nsl_kdd_url, header=None, names=columns)

print(f"NSL-KDD Loaded. Shape: {df_nsl.shape}")

# 3. Clean & Drop the Extra Column
df_nsl.drop('difficulty_level', axis=1, inplace=True)
df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')

# 4. Prepare X (Features) and y (Target)
X_nsl = df_nsl.drop(['label', 'category'], axis=1)
y_nsl_cat = df_nsl['category']

# 5. One-Hot Encoding
X_nsl_encoded = pd.get_dummies(X_nsl, columns=categorical_cols, drop_first=True)

# --- CRITICAL STEP: Feature Alignment & Selection ---
# FIX: Align to X_encoded (the original dataframe) instead of X_train (the numpy array)
try:
    # 1. Add missing columns (fill with 0) and remove extra columns to match training data
    X_nsl_aligned = X_nsl_encoded.reindex(columns=X_encoded.columns, fill_value=0)
    
    # 2. Apply Feature Selection (transform to selected columns only)
    # The 'selection' object remembers which columns we kept during training
    if 'selection' in locals():
        print("Applying Feature Selection filter to test data...")
        X_nsl_final = selection.transform(X_nsl_aligned)
    else:
        X_nsl_final = X_nsl_aligned.values

    print(f"Aligned Test Shape: {X_nsl_final.shape}")

    # 6. Predict
    y_pred_nsl_enc = clf_xgb.predict(X_nsl_final)
    y_pred_nsl_cat = le_cat.inverse_transform(y_pred_nsl_enc)

    # 7. Evaluation
    acc_nsl = accuracy_score(y_nsl_cat, y_pred_nsl_cat)
    print(f"\n>>> Accuracy on NSL-KDD: {acc_nsl:.4f}")
    print("\nClassification Report (NSL-KDD):")
    print(classification_report(y_nsl_cat, y_pred_nsl_cat))

except NameError as e:
    print(f"Error: {e}")
    print("Make sure you have run the 'Preprocessing' and 'Task 1' cells first!")

In [ ]:
# %%
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score

# --- 1. Variable Mapping (The Fix) ---
# We map your existing variables to what the plot expects
try:
    # KDD99 Data (From your original code)
    # Ensure these variables exist from previous cells!
    y_test_data = y_cat_test 
    y_pred_data = y_pred_cat
    
    # NSL-KDD Data (From the validation step)
    # If you named them differently, update these two lines:
    y_nsl_data = y_nsl_cat 
    
    # Smart Check: Did you run the NSL prediction cell?
    if 'y_pred_nsl_cat' in locals():
        y_pred_nsl_data = y_pred_nsl_cat
    elif 'y_pred_nsl_enc' in locals():
         # Recover if only encoded prediction exists
        y_pred_nsl_data = le_cat.inverse_transform(y_pred_nsl_enc)
    else:
        # Fallback: try to grab the last prediction made using the model
        print("Warning: 'y_pred_nsl_cat' not found. Re-running prediction on X_nsl_final...")
        y_pred_nsl_data = le_cat.inverse_transform(clf_xgb.predict(X_nsl_final))

    # Model & Encoder
    model = clf_xgb
    encoder = le_cat
    
    # Feature Names Logic (Handling NumPy Array vs DataFrame)
    if hasattr(X_train, 'columns'):
        feature_names = X_train.columns
    else:
        # If X_train is a numpy array (from Feature Selection), we need to recover names
        # We assume 'selection' object exists from the feature selection step
        if 'selection' in locals() and 'X_encoded' in locals():
             feature_names = X_encoded.columns[selection.get_support()]
        else:
             # Last resort: Generic names
             feature_names = [f"Feat_{i}" for i in range(X_train.shape[1])]
    
    print("Variables mapped successfully!")

except NameError as e:
    print(f"Wait! We are still missing a variable: {e}")
    print("Make sure you have run the 'Training' cell and the 'NSL-KDD' cell above this one.")
    raise e

# --- 2. Plotting Logic ---
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

def plot_side_by_side_matrices(y_true_1, y_pred_1, title_1, y_true_2, y_pred_2, title_2, classes):
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # Plot 1: Baseline
    cm1 = confusion_matrix(y_true_1, y_pred_1, labels=classes)
    sns.heatmap(cm1, annot=True, fmt='d', cmap='Greens', cbar=False, 
                xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_title(f"BASELINE: {title_1}\n(Accuracy: {accuracy_score(y_true_1, y_pred_1):.2%})", 
                      fontsize=14, fontweight='bold', color='#2d6a4b')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')
    
    # Plot 2: Stress Test
    cm2 = confusion_matrix(y_true_2, y_pred_2, labels=classes)
    sns.heatmap(cm2, annot=True, fmt='d', cmap='Reds', cbar=False, 
                xticklabels=classes, yticklabels=classes, ax=axes[1])
    axes[1].set_title(f"STRESS TEST: {title_2}\n(Accuracy: {accuracy_score(y_true_2, y_pred_2):.2%})", 
                      fontsize=14, fontweight='bold', color='#c0392b')
    axes[1].set_yticks([]) # Clean look
    axes[1].set_xlabel('Predicted')
    
    plt.tight_layout()
    plt.show()

# Run Plot
plot_side_by_side_matrices(
    y_test_data, y_pred_data, "Original KDD99", 
    y_nsl_data, y_pred_nsl_data, "NSL-KDD (Hard)", 
    classes=encoder.classes_
)

# --- 3. Feature Importance ---
plt.figure(figsize=(12, 6))
importances = pd.Series(model.feature_importances_, index=feature_names)
top_features = importances.nlargest(12)
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title('Top 12 Features driving your AI', fontsize=15)
plt.show()

# --- 4. Executive Summary ---
acc_base = accuracy_score(y_test_data, y_pred_data)
acc_real = accuracy_score(y_nsl_data, y_pred_nsl_data)
gap = acc_base - acc_real

print("-" * 50)
print(f"BASELINE ACCURACY:   {acc_base:.2%}")
print(f"REAL-WORLD ACCURACY: {acc_real:.2%}")
print(f"OVERFITTING GAP:     {gap:.2%}")
print("-" * 50)

In [ ]:
# %% [markdown]
# ## Phase 3: Real-World Validation & Robustness Upgrade
# In this phase, we address two critical issues:
# 1. **The "Reality Gap":** We test the model on the harder NSL-KDD dataset.
# 2. **Overfitting:** We retrain XGBoost with regularization parameters to improve generalizability.

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --- 1. Configuration: The "Robust" Hyperparameters ---
# These settings force the AI to learn broad concepts instead of memorizing noise.
ROBUST_PARAMS = {
    'n_estimators': 200,        # More trees, but learning slower
    'learning_rate': 0.05,      # Slow down learning to find better patterns
    'max_depth': 3,             # Shallower trees prevent memorizing specific rows
    'min_child_weight': 2,      # Require more data to create a new branch
    'gamma': 1,                 # Pruning: Don't split unless loss drops significantly
    'subsample': 0.8,           # distinct rows: Randomly sample 80% of data per tree
    'colsample_bytree': 0.8,    # distinct columns: Randomly sample 80% of features
    'reg_lambda': 1.0,          # L2 Regularization (Smooths the weights)
    'objective': 'multi:softprob',
    'num_class': len(le_cat.classes_),
    'n_jobs': -1,
    'random_state': 42
}

print("--- Step 1: Training Robust XGBoost Model ---")
clf_robust = xgb.XGBClassifier(**ROBUST_PARAMS)
clf_robust.fit(X_train, y_cat_train_enc)

# Predict on Training Data (KDD99) for Baseline
y_pred_robust_enc = clf_robust.predict(X_test)
y_pred_robust_cat = le_cat.inverse_transform(y_pred_robust_enc)
acc_train = accuracy_score(y_cat_test, y_pred_robust_cat)
print(f" > Robust Training Accuracy (KDD99): {acc_train:.2%}")

# --- 2. Data Loading: NSL-KDD (The Hard Test) ---
print("\n--- Step 2: Loading & aligning NSL-KDD Dataset ---")
NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"

# Define columns manually because the raw file has no header
NSL_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

# Load and Clean
try:
    df_nsl = pd.read_csv(NSL_URL, header=None, names=NSL_COLS)
    df_nsl.drop('difficulty_level', axis=1, inplace=True) # Drop extra column
    df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
    # Ensure attack_map is available from previous cells
    df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')

    # Prepare Features
    X_nsl = df_nsl.drop(['label', 'category'], axis=1)
    y_nsl_cat = df_nsl['category']

    # Align Features (Crucial for Transfer Learning)
    # Ensure categorical_cols is available
    X_nsl_encoded = pd.get_dummies(X_nsl, columns=categorical_cols, drop_first=True)
    
    # Reindex ensures X_nsl has exactly the same columns as X_train, in the same order
    # Handle both DataFrame (with columns) and Numpy Array (without columns) cases for X_train
    if hasattr(X_train, 'columns'):
        train_cols = X_train.columns
    elif 'X_encoded' in locals() and 'selection' in locals():
        # Reconstruct columns if X_train was reduced by selection
        train_cols = X_encoded.columns[selection.get_support()]
    else:
        # Fallback if we can't recover names (rare)
        print("Warning: Could not recover original column names for alignment. Validation might fail.")
        # We have to guess using the NSL columns, but this is risky if they differ
        train_cols = X_nsl_encoded.columns 

    X_nsl_final = X_nsl_encoded.reindex(columns=train_cols, fill_value=0)

    # Predict on NSL-KDD
    y_pred_nsl_enc = clf_robust.predict(X_nsl_final)
    y_pred_nsl_cat = le_cat.inverse_transform(y_pred_nsl_enc)
    acc_nsl = accuracy_score(y_nsl_cat, y_pred_nsl_cat)
    print(f" > Robust Test Accuracy (NSL-KDD): {acc_nsl:.2%}")

except Exception as e:
    print(f"Error loading/processing NSL-KDD data: {e}")
    # Set dummy values so report doesn't crash if load fails
    acc_nsl = 0 
    y_nsl_cat = []
    y_pred_nsl_cat = []


# --- 3. Advanced Visualization ---
print("\n--- Step 3: Generating Executive Report ---")

def plot_comparison_matrices(y_true1, y_pred1, title1, y_true2, y_pred2, title2, classes):
    if len(y_true2) == 0:
        print("Skipping plots: No test data available.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1 (Green Theme for Baseline)
    sns.heatmap(confusion_matrix(y_true1, y_pred1, labels=classes), 
                annot=True, fmt='d', cmap='Greens', cbar=False, 
                xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_title(f"{title1}\nAcc: {accuracy_score(y_true1, y_pred1):.2%}", fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')

    # Plot 2 (Red Theme for Stress Test)
    sns.heatmap(confusion_matrix(y_true2, y_pred2, labels=classes), 
                annot=True, fmt='d', cmap='Reds', cbar=False, 
                xticklabels=classes, yticklabels=classes, ax=axes[1])
    axes[1].set_title(f"{title2}\nAcc: {accuracy_score(y_true2, y_pred2):.2%}", fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicted')
    axes[1].set_yticks([]) 

    plt.tight_layout()
    plt.show()

# Only plot if we successfully loaded data
if acc_nsl > 0:
    plot_comparison_matrices(
        y_cat_test, y_pred_robust_cat, "Baseline (KDD99)",
        y_nsl_cat, y_pred_nsl_cat, "Stress Test (NSL-KDD)",
        classes=le_cat.classes_
    )

# --- 4. Feature Importance ---
# Handle NumPy array case for X_train
try:
    if hasattr(X_train, 'columns'):
        feat_names = X_train.columns
    elif 'X_encoded' in locals() and 'selection' in locals():
        feat_names = X_encoded.columns[selection.get_support()]
    else:
        feat_names = [f"F{i}" for i in range(X_train.shape[1])]

    plt.figure(figsize=(10, 5))
    importances = pd.Series(clf_robust.feature_importances_, index=feat_names)
    importances.nlargest(10).plot(kind='barh', color='#4a148c')
    plt.title('Top 10 Features (Robust Model)')
    plt.show()
except Exception as e:
    print(f"Skipping feature importance plot: {e}")

# --- 5. Final Scorecard ---
gap = acc_train - acc_nsl
print("-" * 40)
print("FINAL ROBUSTNESS SCORECARD")
print("-" * 40)
print(f"Baseline Accuracy:   {acc_train:.2%}")
print(f"Real-World Accuracy: {acc_nsl:.2%}")
print(f"Generalization Gap:  {gap:.2%}")

if gap < 0.15:
    print("✅ STATUS: PASSED. The model is robust.")
else:
    print("⚠️ STATUS: WARNING. High overfitting detected.")
print("-" * 40)

In [ ]:
# %% [markdown]
# ## Phase 4: Ensemble Learning (The "Nuclear Option")
# To fix the accuracy gap, we stop relying on a single model.
# We build a **Voting Classifier** that averages the decisions of three different algorithms.
# This smoothes out the biases of any single model.

# %%
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("--- Building the Council of Experts (Ensemble Model) ---")

# 1. expert: XGBoost (The Specialist)
# High depth, learns complex patterns
clf_xgb_expert = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    n_jobs=-1,
    random_state=42
)

# 2. expert: Random Forest (The Generalist)
# Different math (Bagging vs Boosting) helps cover XGBoost's blind spots
clf_rf_expert = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10,  # Restricted depth to prevent memorization
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

# 3. expert: Logistic Regression (The Skeptic)
# Simple linear boundaries. If the other two get too crazy, this pulls them back.
# Needs scaling to work well.
clf_lr_expert = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, n_jobs=-1))

# --- Combine them into the Voting Classifier ---
# 'soft' voting means we average their predicted probabilities, not just their final votes.
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', clf_xgb_expert),
        ('rf', clf_rf_expert),
        ('lr', clf_lr_expert)
    ],
    voting='soft',
    weights=[2, 2, 1]  # Trust XGB and RF twice as much as LR
)

# Train the Ensemble
print("Training the Ensemble (this may take a moment)...")
voting_clf.fit(X_train, y_cat_train_enc)

# --- Evaluate on Real World Data (NSL-KDD) ---
print("\n--- Ensemble Validation ---")

# Predict
y_pred_ens_enc = voting_clf.predict(X_nsl_final)
y_pred_ens_cat = le_cat.inverse_transform(y_pred_ens_enc)

# Score
acc_ens = accuracy_score(y_nsl_cat, y_pred_ens_cat)
acc_train_ens = accuracy_score(y_cat_test, le_cat.inverse_transform(voting_clf.predict(X_test)))

print(f"Baseline Training Accuracy:   {acc_train_ens:.2%}")
print(f"Real-World (NSL) Accuracy:    {acc_ens:.2%}")
print(f"New Generalization Gap:       {acc_train_ens - acc_ens:.2%}")

print("\nClassification Report (Ensemble):")
print(classification_report(y_nsl_cat, y_pred_ens_cat))

# --- Update the main model variable for plotting later ---
clf_xgb = voting_clf 
# Note: Feature importance plots won't work directly on VotingClassifier 
# because it doesn't have a single set of features. We skip that plot for this step.

In [ ]:
# %% [markdown]
# ## Phase 5: The Grand Finale (Robust Training & Visualization)
# This block loads the hard validation data, balances the training set, 
# trains the "Council of Experts" ensemble, and visualizes the final results.

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# --- PART 1: Load NSL-KDD (The "Hard" Test Set) ---
print("--- 1. Loading & Aligning NSL-KDD Dataset ---")
NSL_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"
NSL_COLS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
    'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]

try:
    df_nsl = pd.read_csv(NSL_URL, header=None, names=NSL_COLS)
    df_nsl.drop('difficulty_level', axis=1, inplace=True)
    df_nsl['label'] = df_nsl['label'].astype(str).str.replace('.', '', regex=False)
    df_nsl['category'] = df_nsl['label'].map(attack_map).fillna('other')
    
    X_nsl = df_nsl.drop(['label', 'category'], axis=1)
    y_nsl_cat = df_nsl['category']
    
    # One-Hot Encode (same process as training)
    X_nsl_encoded = pd.get_dummies(X_nsl, columns=categorical_cols, drop_first=True)
    
    # FEATURE RECOVERY & ALIGNMENT
    # We need to ensure the test data has the exact same columns as the training data
    if hasattr(X_train, 'columns'):
        train_cols = X_train.columns
    elif 'selection' in locals() and 'X_encoded' in locals():
        # If X_train is a numpy array from selection, recover names using the mask
        train_cols = X_encoded.columns[selection.get_support()]
    else:
        # Fallback: This implies Feature Selection wasn't run or variables are lost
        print("Warning: Could not determine training columns. Using intersection.")
        train_cols = X_nsl_encoded.columns

    # Align columns (fill missing with 0, drop extra)
    X_nsl_final = X_nsl_encoded.reindex(columns=train_cols, fill_value=0)
    print(f"NSL-KDD Loaded. Shape: {X_nsl_final.shape}")

except Exception as e:
    print(f"Error loading NSL-KDD: {e}")
    X_nsl_final = None

# --- PART 2: Synthetic Oversampling (Fixing Class Imbalance) ---
print("\n--- 2. Performing Manual Synthetic Oversampling ---")

if hasattr(X_train, 'columns'):
    # X_train is a DataFrame
    train_df = X_train.copy()
    train_df['target'] = y_cat_train_enc
else:
    # X_train is a NumPy array (from Feature Selection)
    train_df = pd.DataFrame(X_train, columns=train_cols)
    train_df['target'] = y_cat_train_enc

class_counts = train_df['target'].value_counts()
rare_classes = class_counts[class_counts < 1000].index.tolist()

oversampled_data = [train_df]

for cls in rare_classes:
    rare_samples = train_df[train_df['target'] == cls]
    if len(rare_samples) == 0: continue
    
    multiplier = 2000 // len(rare_samples)
    if multiplier > 1:
        print(f"Oversampling Class {cls} (x{multiplier})...")
        for _ in range(multiplier):
            synthetic = rare_samples.copy()
            # Add noise to numeric columns only
            numeric_cols = ['duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count']
            valid_cols = [c for c in numeric_cols if c in synthetic.columns]
            if valid_cols:
                noise = np.random.normal(0, 0.01, synthetic[valid_cols].shape)
                synthetic[valid_cols] += noise
            oversampled_data.append(synthetic)

train_df_balanced = pd.concat(oversampled_data)
train_df_balanced = shuffle(train_df_balanced, random_state=42)
print("New Class Distribution:\n", train_df_balanced['target'].value_counts())

X_train_bal = train_df_balanced.drop('target', axis=1)
y_train_bal = train_df_balanced['target']

# --- PART 3: Train Ensemble Model ---
print("\n--- 3. Training Council of Experts (Ensemble) ---")

clf_xgb_final = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, objective='multi:softprob', n_jobs=-1, random_state=42)
clf_rf_final = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42)
clf_lr_final = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1))

voting_clf = VotingClassifier(
    estimators=[('xgb', clf_xgb_final), ('rf', clf_rf_final), ('lr', clf_lr_final)],
    voting='soft'
)

voting_clf.fit(X_train_bal, y_train_bal)
print("Ensemble Training Complete.")

# Update global model for other plots if needed
clf_xgb = voting_clf 

# --- PART 4: Final Evaluation & Visualization ---
print("\n--- 4. Final Results ---")

# Predictions
if X_nsl_final is not None:
    y_pred_ens_enc = voting_clf.predict(X_nsl_final)
    y_pred_ens_cat = le_cat.inverse_transform(y_pred_ens_enc)
    
    # Metrics
    acc_final = accuracy_score(y_nsl_cat, y_pred_ens_cat)
    print(f"Final Real-World Accuracy (NSL-KDD): {acc_final:.2%}")
    print("\nClassification Report:")
    print(classification_report(y_nsl_cat, y_pred_ens_cat))
    
    # --- Visualization Code ---
    # Compare Baseline (KDD99) vs Real-World (NSL-KDD)
    
    # Get predictions on original test set for baseline
    y_pred_base_enc = voting_clf.predict(X_test) # X_test is KDD99
    y_pred_base_cat = le_cat.inverse_transform(y_pred_base_enc)
    acc_base = accuracy_score(y_cat_test, y_pred_base_cat)
    
    labels = le_cat.classes_
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Baseline (Easy)
    sns.heatmap(confusion_matrix(y_cat_test, y_pred_base_cat, labels=labels), 
                annot=True, fmt='d', cmap='Greens', cbar=False, 
                xticklabels=labels, yticklabels=labels, ax=axes[0])
    axes[0].set_title(f"Training Environment (KDD99)\nAccuracy: {acc_base:.2%}", fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')

    # Plot 2: Real World (Hard)
    sns.heatmap(confusion_matrix(y_nsl_cat, y_pred_ens_cat, labels=labels), 
                annot=True, fmt='d', cmap='Reds', cbar=False, 
                xticklabels=labels, yticklabels=labels, ax=axes[1])
    axes[1].set_title(f"Real-World Stress Test (NSL-KDD)\nAccuracy: {acc_final:.2%}", fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicted')
    axes[1].set_yticks([]) 
    
    plt.tight_layout()
    plt.show()
    
    print("-" * 40)
    print(f"Generalization Gap: {acc_base - acc_final:.2%}")
    print("-" * 40)
else:
    print("Could not run validation due to data loading errors.")

# 